# Deterministic linkage rules

A deterministic rule states, in advance, which fields two records must agree on
for them to be treated as the same person. This notebook builds five such rules
on the bundled registers, from the strictest to the most permissive, and scores
each one against the known truth.

The point is not that one rule wins. It is that each rule sits somewhere on a
trade-off between finding matches and being right about them, and that you can
see exactly where — which is what makes a deterministic design defensible.

**What you will do**

1. Build a strict rule and measure it
2. Relax it to an N-1 rule and see what you gain and lose
3. Add a non-disagreement clause and see the difference it makes
4. Try a match-key rule built from fragments of fields
5. Combine rules into a stepwise design
6. Compare all five, and check whether the results respect the expected 1:1
   structure

## 0. Setup and prepared data

The same preparation as [chapter 2.2](nb01-inspect-and-prepare.ipynb).

In [1]:
import unicodedata
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 130)

DATA = Path("../../data")
if not DATA.exists():
    DATA = Path("data")

fonasa = pd.read_csv(DATA / "fonasa_sample.csv", dtype=str)
suseso = pd.read_csv(DATA / "suseso_sample.csv", dtype=str)


def basic_text_clean(series):
    return (series.astype("string").str.strip().str.upper()
            .str.replace(r"\s+", " ", regex=True))


def remove_accents(value):
    if pd.isna(value):
        return pd.NA
    value = unicodedata.normalize("NFKD", str(value))
    return "".join(c for c in value if not unicodedata.combining(c))


def standardise_name(series):
    cleaned = basic_text_clean(series)
    cleaned = cleaned.map(remove_accents, na_action="ignore").astype("string")
    cleaned = cleaned.str.replace(r"[^A-ZN ]", "", regex=True)
    return cleaned.str.replace(r"\s+", " ", regex=True).str.strip()


def clean_sex(series):
    cleaned = basic_text_clean(series)
    return cleaned.replace({"HOMBRE": "M", "MUJER": "F", "MASCULINO": "M",
                            "FEMENINO": "F", "": pd.NA})


for df in (fonasa, suseso):
    for col in ["nombre", "ap1", "ap2"]:
        df[f"{col}_clean"] = standardise_name(df[col])
    df["sexo_clean"] = clean_sex(df["sexo"])
    df["nac_clean"] = basic_text_clean(df["nacionalidad"])

CLEAN = ["nombre_clean", "ap1_clean", "ap2_clean", "sexo_clean", "nac_clean"]
TRUE_MATCHES = 4500

print(f"fonasa: {len(fonasa):,} records | suseso: {len(suseso):,} records")
print(f"true matches present in the data: {TRUE_MATCHES:,}")

fonasa: 30,000 records | suseso: 27,000 records
true matches present in the data: 4,500


## 1. Scoring machinery

Every rule in this notebook produces a set of candidate pairs. To compare them
we need one function that scores any such set against the truth.

Three measures, and they answer different questions:

- **Precision** — of the pairs the rule proposed, what share are correct? This
  is the risk of a false match.
- **Recall** — of the 4,500 real matches, what share did the rule find? This is
  the risk of a missed match.
- **F1** — their harmonic mean, a single number for when you weight the two
  errors equally. If you do not weight them equally, do not use it.

In [2]:
def score(pairs, label):
    """Score a set of candidate pairs against the known truth."""
    n = len(pairs)
    tp = int((pairs["true_person_id_f"] == pairs["true_person_id_s"]).sum())
    fp = n - tp
    precision = tp / n if n else float("nan")
    recall = tp / TRUE_MATCHES
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return {
        "rule": label,
        "pairs_proposed": n,
        "correct": tp,
        "incorrect": fp,
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
    }


def apply_rule(left_fields, right_fields=None, require=None, label=""):
    """Join the two registers on an exact-agreement key built from `left_fields`.

    `require` lists the fields that must be non-missing for a record to be
    eligible at all; a record missing any of them is dropped before the join.
    """
    right_fields = right_fields or left_fields
    require = require or left_fields

    l = fonasa.dropna(subset=require).copy()
    r = suseso.dropna(subset=require).copy()
    l["_key"] = l[left_fields].agg("|".join, axis=1)
    r["_key"] = r[right_fields].agg("|".join, axis=1)

    keep = ["unique_id", "_key", "true_person_id"] + CLEAN
    pairs = l[keep].merge(r[keep], on="_key", suffixes=("_f", "_s"))
    return pairs

## 2. Rule 1 — strict

All five identifying fields must be present and must agree exactly. This is the
most conservative rule available, and it is the same logic as the hashed
identifier in [chapter 2.3](nb02-pseudonymisation.ipynb).

In [3]:
r1 = apply_rule(CLEAN, label="R1")
results = [score(r1, "R1 strict: all five fields agree")]
pd.DataFrame(results)

,rule,pairs_proposed,correct,incorrect,precision,recall,f1
0,R1 strict: all five fields agree,1067,1067,0,1.0,0.2371,0.3833


As expected: nearly everything it proposes is correct, and it finds a small
minority of the true matches.

The reason is worth restating, because it is the reason deterministic rules get
relaxed at all. A strict rule fails on any record with a missing required field,
and on any record with a single character of disagreement anywhere. Those are
not rare events in administrative data.

## 3. Rule 2 — N-1

An **N-1 rule** requires agreement on all but one of the fields. Here we drop
the second surname from the requirement: it is the most frequently missing name
field, and in many countries it is recorded inconsistently or not at all.

The record no longer has to *have* a second surname, and the second surname is
no longer compared.

In [4]:
N_MINUS_1 = ["nombre_clean", "ap1_clean", "sexo_clean", "nac_clean"]

r2 = apply_rule(N_MINUS_1, label="R2")
results.append(score(r2, "R2 N-1: drop ap2 from the rule"))
pd.DataFrame(results)

,rule,pairs_proposed,correct,incorrect,precision,recall,f1
0,R1 strict: all five fields agree,1067,1067,0,1.0000,0.2371,0.3833
1,R2 N-1: drop ap2 from the rule,1197,1113,84,0.9298,0.2473,0.3907


Recall goes up, and precision comes down. That is the trade in its purest form:
the rule now accepts pairs it previously rejected, and some of those pairs are
different people who happen to share a given name, a first surname, a sex and a
nationality.

## 4. Rule 3 — N-1 with a non-disagreement clause

Rule 2 throws away information. It ignores the second surname entirely, even
when both records have one and the two differ — which is good evidence that they
are different people.

A **non-disagreement clause** uses that evidence without requiring the field to
be present. The field is not required to agree; it is only required not to
conflict.

- both values present and equal → accept
- either value missing → accept
- both present and different → reject

In [5]:
both_present = r2["ap2_clean_f"].notna() & r2["ap2_clean_s"].notna()
conflict = both_present & (r2["ap2_clean_f"] != r2["ap2_clean_s"])

r3 = r2[~conflict].copy()
results.append(score(r3, "R3 N-1 + non-disagreement on ap2"))

print(f"pairs rejected by the clause: {conflict.sum():,}")
print(f"  of which were actually true matches: "
      f"{int((r2.loc[conflict, 'true_person_id_f'] == r2.loc[conflict, 'true_person_id_s']).sum()):,}")
pd.DataFrame(results)

pairs rejected by the clause: 107
  of which were actually true matches: 26


,rule,pairs_proposed,correct,incorrect,precision,recall,f1
0,R1 strict: all five fields agree,1067,1067,0,1.0000,0.2371,0.3833
1,R2 N-1: drop ap2 from the rule,1197,1113,84,0.9298,0.2473,0.3907
2,R3 N-1 + non-disagreement on ap2,1090,1087,3,0.9972,0.2416,0.3889


This is the most useful single trick in deterministic linkage. The clause
removes pairs at a favourable ratio — roughly three wrong pairs for every right
one it costs — so precision recovers almost to the strict rule's level while
recall stays above it.

Non-disagreement clauses are worth considering for any field that is often
missing but reliable when present.

## 5. Rule 4 — a match key

A **match key** builds a rule out of *fragments* of fields rather than whole
ones. It is a way of tolerating error in a specific, predictable place.

Here: the first three letters of the given name, plus both surnames and sex.
This survives a person recorded as `JOSE` in one system and `JOSEFINA` in the
other, and it survives most typographical errors in the tail of a name — but it
insists on both surnames, which makes it strict in a different direction.

In [6]:
for df in (fonasa, suseso):
    df["nombre_3"] = df["nombre_clean"].str[:3]

MATCH_KEY = ["nombre_3", "ap1_clean", "ap2_clean", "sexo_clean"]

r4 = apply_rule(MATCH_KEY, label="R4")
results.append(score(r4, "R4 match key: nombre[:3] + ap1 + ap2 + sexo"))
pd.DataFrame(results)

,rule,pairs_proposed,correct,incorrect,precision,recall,f1
0,R1 strict: all five fields agree,1067,1067,0,1.0000,0.2371,0.3833
1,R2 N-1: drop ap2 from the rule,1197,1113,84,0.9298,0.2473,0.3907
2,R3 N-1 + non-disagreement on ap2,1090,1087,3,0.9972,0.2416,0.3889
3,R4 match key: nombre[:3] + ap1 + ap2 + sexo,1158,1135,23,0.9801,0.2522,0.4012


A different point on the same trade-off, reached by relaxing a different thing.

The interesting question is whether these rules are **nested** — whether the
permissive ones simply contain the strict one — or whether each finds pairs the
others do not. That decides whether combining them is worth anything.

In [7]:
def pair_set(pairs):
    return set(zip(pairs["unique_id_f"], pairs["unique_id_s"]))


s1, s3, s4 = pair_set(r1), pair_set(r3), pair_set(r4)

print(f"R1 strict            : {len(s1):,} pairs")
print(f"R3 N-1 + clause      : {len(s3):,} pairs")
print(f"R4 match key         : {len(s4):,} pairs")
print()
print(f"found by R3 but not R1: {len(s3 - s1):,}")
print(f"found by R1 but not R3: {len(s1 - s3):,}")
print(f"found by R4 but not R1: {len(s4 - s1):,}")
print(f"found by R1 but not R4: {len(s1 - s4):,}")
print(f"found by R4 but not R3: {len(s4 - s3):,}")
print(f"found by R3 but not R4: {len(s3 - s4):,}")
print()
print(f"union of all three    : {len(s1 | s3 | s4):,} pairs")

R1 strict            : 1,067 pairs
R3 N-1 + clause      : 1,090 pairs
R4 match key         : 1,158 pairs

found by R3 but not R1: 23
found by R1 but not R3: 0
found by R4 but not R1: 91
found by R1 but not R4: 0
found by R4 but not R3: 91
found by R3 but not R4: 23

union of all three    : 1,181 pairs


The answer is partly one and partly the other, and both halves are useful.

The strict rule is **contained** in both of the others: every pair R1 finds, R3
and R4 also find. Relaxing a requirement never loses a pair that met the stricter
one, which is reassuring but not surprising.

R3 and R4, on the other hand, are **not** nested in each other. R4 finds 91 pairs
R3 does not; R3 finds 23 that R4 does not. Neither is a relaxation of the other:
one tolerates a missing second surname, the other tolerates a wrong ending on a
given name, and those are different failures. Their union, 1,181 pairs, is larger
than either alone.

That is what motivates the next rule: if rules catch different errors, run
several.

## 6. Rule 5 — stepwise

A **stepwise** design applies rules in sequence, from strict to permissive, and
each stage only sees the records that earlier stages did not match. Every pair
carries a label saying which stage produced it, which means the strong and weak
parts of the linkage stay distinguishable afterwards.

In [8]:
def stepwise(stages):
    """Apply rules in order; each stage only sees records not yet matched."""
    used_f, used_s = set(), set()
    out = []
    for label, pairs in stages:
        fresh = pairs[~pairs["unique_id_f"].isin(used_f)
                      & ~pairs["unique_id_s"].isin(used_s)].copy()
        fresh["stage"] = label
        used_f |= set(fresh["unique_id_f"])
        used_s |= set(fresh["unique_id_s"])
        out.append(fresh)
    return pd.concat(out, ignore_index=True)


r5 = stepwise([("1 strict", r1), ("2 N-1 + clause", r3), ("3 match key", r4)])
results.append(score(r5, "R5 stepwise: R1, then R3, then R4"))

print("pairs contributed by each stage:")
print(r5["stage"].value_counts().sort_index().to_string())
print()
print("precision within each stage:")
print(r5.groupby("stage")
        .apply(lambda g: round((g["true_person_id_f"] == g["true_person_id_s"]).mean(), 4),
               include_groups=False)
        .to_string())

pairs contributed by each stage:
stage
1 strict          1067
2 N-1 + clause      23
3 match key         87

precision within each stage:
stage
1 strict          1.0000
2 N-1 + clause    0.8696
3 match key       0.7816


The per-stage precision is the payoff of a stepwise design. Stage 1 is close to
certain; later stages are progressively less so, and you know which pairs came
from where.

That matters operationally. An analyst can be told that stage 1 pairs are safe
to use unconditionally while stage 3 pairs carry a known error rate, or can drop
the weakest stage entirely for an analysis that needs high precision — without
re-running anything.

## 7. All five rules together

In [9]:
comparison = pd.DataFrame(results)
comparison

,rule,pairs_proposed,correct,incorrect,precision,recall,f1
0,R1 strict: all five fields agree,1067,1067,0,1.0000,0.2371,0.3833
1,R2 N-1: drop ap2 from the rule,1197,1113,84,0.9298,0.2473,0.3907
2,R3 N-1 + non-disagreement on ap2,1090,1087,3,0.9972,0.2416,0.3889
3,R4 match key: nombre[:3] + ap1 + ap2 + sexo,1158,1135,23,0.9801,0.2522,0.4012
4,"R5 stepwise: R1, then R3, then R4",1177,1155,22,0.9813,0.2567,0.4069


Read the table as a curve rather than a ranking. Moving down it, recall rises and
precision falls, and no rule dominates the others.

Which point you want is not a property of the data. It is the decision you
recorded in [chapter 2.1](defining-the-use-case.md): whether a false match or a
missed match is more costly in your use case.

## 8. Does the result respect the expected structure?

The specification said this link is 1:1 — each person should appear at most once
on each side. Nothing in a deterministic rule enforces that, so check it.

This is one of the few quality checks you can run without a ground truth, which
makes it valuable on real data.

In [10]:
for label, pairs in [("R1 strict", r1), ("R3 N-1 + clause", r3), ("R4 match key", r4)]:
    f_counts = pairs["unique_id_f"].value_counts()
    s_counts = pairs["unique_id_s"].value_counts()
    print(f"{label}:")
    print(f"  health-register records matched to more than one: {(f_counts > 1).sum():,}"
          f"  (max {f_counts.max()})")
    print(f"  social-security records matched to more than one: {(s_counts > 1).sum():,}"
          f"  (max {s_counts.max()})")

R1 strict:
  health-register records matched to more than one: 0  (max 1)
  social-security records matched to more than one: 0  (max 1)
R3 N-1 + clause:
  health-register records matched to more than one: 0  (max 1)
  social-security records matched to more than one: 0  (max 1)
R4 match key:
  health-register records matched to more than one: 4  (max 2)
  social-security records matched to more than one: 3  (max 2)


Where a record matches several others, the rule key is not unique enough to
identify a person: several different people share that combination of values. In
a 1:1 link those pairs need resolution — either by a tie-break rule, or by
clerical review, or by tightening the rule.

A rule that produces many multiple matches is telling you something. In a larger
register, a rule that looks fine here would produce far more of them, because the
number of coincidental agreements grows with the size of the file while the
number of true matches does not.

## 9. Where deterministic linkage runs out

Everything above shares one structural property: a rule makes a **binary**
decision from an **exact** comparison. That has three consequences.

**Partial agreement is invisible.** `MARIA JOSE` and `MARIA JOSÉ` after cleaning
may agree; `MARIA JOSE` and `MARIA` do not, and the rule cannot express "these
are 80% similar".

**All agreements weigh the same.** Two records agreeing on the surname
`GONZALEZ` — the most common surname in both registers — counts exactly as much
as agreement on a surname held by three people in the country. Intuitively the
second is far stronger evidence. A deterministic rule has no way to say so.

**The trade-off is coarse.** You move along the curve by adding or removing whole
rules, which is a blunt instrument, and each new rule has to be designed, tested
and documented by hand.

The Fellegi-Sunter framework addresses all three by replacing the binary decision
with a score, and by deriving the weight of each field's agreement from the data
rather than from judgement.
[The next notebook](nb04-fellegi-sunter-by-hand.ipynb) builds that framework from
first principles, using nothing but the counts you can compute from these same
two files.